⇒ 과제 목표: 혐오 표현 여부 (hate: true/false) 분류

label : 혐오 유형 (0~7 → 혐오, 8 → 비혐오)

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch.nn.functional as F
from tqdm import tqdm

In [ ]:
# 1. 데이터 불러오기
url = "https://raw.githubusercontent.com/adlnlp/K-MHaS/refs/heads/main/data/kmhas_train.txt"
df = pd.read_csv(url, sep="\t")

In [ ]:
# 2. 여러 개 라벨 중 맨 앞의 라벨 하나만 사용
df["label"] = df["label"].apply(lambda x: int(str(x).split(",")[0]))

# 3. 라벨 매핑
num_labels = 9  # 0~8
label2id = {i: i for i in range(num_labels)}
id2label = {i: str(i) for i in range(num_labels)}

print("라벨 분포:\n", df["label"].value_counts())

라벨 분포:
 label
8    42909
3     7773
2     7352
0     7272
1     6177
4     3565
5     2454
7     1352
6      123
Name: count, dtype: int64


In [ ]:
# 4. train/val split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["document"], df["label"], test_size=0.1, random_state=42
)

# 4-2) 위에서 만든 함수 재사용(Train에서만 8 삭제)
import numpy as np
import pandas as pd
from collections import Counter

def drop_excess_class8_from_split(train_texts, train_labels, target="median_non8", random_state=42, shuffle=True):
    """
    train_texts, train_labels: train_test_split 결과 (Series/ndarray 모두 OK)
    target:
      - "median_non8": 8의 개수를 '8이 아닌 클래스'들의 중앙값까지 줄임(권장)
      - 0~1 사이 실수 (예: 0.5): 8을 해당 비율만 남김(절반 등)
      - 양의 정수 (예: 1200): 8을 딱 그 개수만 남김
    반환: (train_texts_bal, train_labels_bal)
    """
    df_tr = pd.DataFrame({"text": train_texts, "label": train_labels})
    s = df_tr["label"]

    # 현재 라벨 분포
    counts = s.value_counts(dropna=False)
    cnt8 = int(counts.get(8, counts.get("8", 0)))
    if cnt8 == 0:
        print("라벨 8이 없어 삭제 없음.")
        return df_tr["text"].reset_index(drop=True), df_tr["label"].reset_index(drop=True)

    # 목표 개수 계산
    counts_non8 = counts.drop(labels=[8, "8"], errors="ignore").values
    if isinstance(target, str) and target == "median_non8":
        target_n_8 = int(np.median(counts_non8)) if len(counts_non8) > 0 else cnt8
    elif isinstance(target, float) and 0 < target <= 1:
        target_n_8 = int(cnt8 * target)
    elif isinstance(target, int) and target > 0:
        target_n_8 = min(target, cnt8)
    else:
        raise ValueError("target은 'median_non8' / (0~1 실수) / 양의 정수 중 하나여야 합니다.")

    if cnt8 <= target_n_8:
        print(f"현재 8의 개수({cnt8})가 목표({target_n_8}) 이하라서 삭제 없음.")
        out = df_tr.copy()
        if shuffle:
            out = out.sample(frac=1, random_state=random_state)
        return out["text"].reset_index(drop=True), out["label"].reset_index(drop=True)

    # 8인 행 중 '남길' 개수만 샘플링(나머지는 삭제)
    is8 = (s.astype(str) == "8")
    idx_8_keep = df_tr[is8].sample(n=target_n_8, random_state=random_state).index
    idx_non8   = df_tr[~is8].index
    keep_idx = idx_non8.union(idx_8_keep)

    before = counts.sort_index()
    out = df_tr.loc[keep_idx]
    if shuffle:
        out = out.sample(frac=1, random_state=random_state)
    out = out.reset_index(drop=True)

    after = out["label"].value_counts(dropna=False).sort_index()
    print("=== Train 분포 변화(삭제 전 -> 삭제 후) ===")
    print(pd.concat([before.rename("before"), after.rename("after")], axis=1).fillna(0).astype(int))

    return out["text"], out["label"]

In [ ]:
# 5. 토크나이저
tokenizer = BertTokenizer.from_pretrained("klue/bert-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

In [ ]:
# 6. Dataset 정의
class HateSpeechDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_len)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# 7. Dataset, DataLoader
train_dataset = HateSpeechDataset(train_texts, train_labels, tokenizer)
val_dataset = HateSpeechDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
# 8. 모델 (멀티클래스 → num_labels=9)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained(
    "klue/bert-base",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
# 9. Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
# 10. 학습 루프
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 4443/4443 [24:54<00:00,  2.97it/s]


Epoch 1 - Average Loss: 0.5340


Epoch 2: 100%|██████████| 4443/4443 [24:55<00:00,  2.97it/s]


Epoch 2 - Average Loss: 0.3647


Epoch 3: 100%|██████████| 4443/4443 [24:55<00:00,  2.97it/s]


Epoch 3 - Average Loss: 0.2361


Epoch 4: 100%|██████████| 4443/4443 [24:55<00:00,  2.97it/s]


Epoch 4 - Average Loss: 0.1506


Epoch 5: 100%|██████████| 4443/4443 [24:54<00:00,  2.97it/s]

Epoch 5 - Average Loss: 0.1027


In [ ]:
# 11. 검증
def evaluate(model, val_loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(F.softmax(logits, dim=1), dim=1)

            preds.extend(predictions.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets, preds)
    print(f"Validation Accuracy: {acc:.4f}")

In [19]:
# 멀티클래스 테스트/추론 코드

import torch

# 0~7 : 혐오유형, 8 : 비혐오
def label_to_text(lbl: int) -> str:
    return "비혐오" if lbl == 8 else f"혐오({lbl})"

def predict_labels(texts, model, tokenizer, device, max_len=128, topk=3):

    model.eval()
    enc = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**enc).logits
        prob = torch.softmax(logits, dim=-1)

    preds = prob.argmax(dim=-1).cpu().tolist()
    probs = prob.cpu().tolist()

    # top-k 정보
    topk_vals, topk_idx = torch.topk(prob, k=min(topk, prob.shape[-1]), dim=-1)
    topk_info = []
    for vals, idxs in zip(topk_vals.cpu(), topk_idx.cpu()):
        topk_info.append([(int(i), float(v)) for i, v in zip(idxs, vals)])

    return preds, probs, topk_info

def pretty_print(texts, preds, probs, topk_info):
    for t, p, pr, tk in zip(texts, preds, probs, topk_info):
        hate_flag = "혐오" if p != 8 else "비혐오"
        print(f"\n{hate_flag} / 라벨={p} ({label_to_text(p)})")
        print(f"문장: {t}")
        tk_str = ", ".join([f"{lbl}({label_to_text(lbl)}): {val:.2f}" for lbl, val in tk])
        print(f"Top-k: {tk_str}")


test_sentences = [
    "진짜 못생기고 한심하다. 거울 좀 봐라.",
    "니 같은 쓰레기랑 말 섞는 것도 아깝다.",
    "쟤는 태어나지 말았어야 했어.",
    "고맙습니다"
]

# ── 추론 실행
preds, probs, topk_info = predict_labels(test_sentences, model, tokenizer, device, max_len=128, topk=3)
pretty_print(test_sentences, preds, probs, topk_info)



혐오 / 라벨=1 (혐오(1))
문장: 진짜 못생기고 한심하다. 거울 좀 봐라.
Top-k: 1(혐오(1)): 1.00, 3(혐오(3)): 0.00, 4(혐오(4)): 0.00

비혐오 / 라벨=8 (비혐오)
문장: 니 같은 쓰레기랑 말 섞는 것도 아깝다.
Top-k: 8(비혐오): 0.47, 0(혐오(0)): 0.36, 7(혐오(7)): 0.11

비혐오 / 라벨=8 (비혐오)
문장: 쟤는 태어나지 말았어야 했어.
Top-k: 8(비혐오): 0.99, 3(혐오(3)): 0.01, 1(혐오(1)): 0.00

비혐오 / 라벨=8 (비혐오)
문장: 고맙습니다
Top-k: 8(비혐오): 1.00, 3(혐오(3)): 0.00, 0(혐오(0)): 0.00
